# Build stac-geoparquet from catalog.json using rustac

This notebook reads the local STAC catalog (produced by `build_dynamical_catalog_full.ipynb`)
and writes a stac-geoparquet file using rustac's own writer.

The goal is to check whether rustac preserves non-standard top-level fields like
`storage:schemes` — which `stac_geoparquet` currently drops — so that
[xpystac](https://github.com/stac-utils/xpystac) can open the resulting items
directly without reconstructing the storage configuration.

In [ ]:
import asyncio
import pystac
import rustac
import xarray as xr
import xpystac  # registers 'stac' xarray backend

CATALOG_URL = "https://r2-pub.openscicomp.io/stac/dynamical/catalog.json"
OUT_PARQUET = "/tmp/dynamical_rustac.parquet"

## 1. Read all items from the JSON catalog

`rustac.read()` fetches a STAC object (catalog, collection, or item) from a URL.
For a catalog, follow the item links manually to collect all items.

In [ ]:
from urllib.parse import urljoin

catalog = await rustac.read(CATALOG_URL)

item_urls = [
    urljoin(CATALOG_URL, link["href"])
    for link in catalog.get("links", [])
    if link.get("rel") == "item"
]
print(f"{len(item_urls)} item links found")
print("Example URL:", item_urls[0])

items = [await rustac.read(url) for url in item_urls]
print(f"\n{len(items)} items loaded")
print("Top-level keys:", list(items[0].keys()))
print("storage:schemes present:", "storage:schemes" in items[0])

## 2. Write to stac-geoparquet using rustac

In [ ]:
async with rustac.geoparquet_writer(items[:1], OUT_PARQUET) as w:
    if len(items) > 1:
        await w.write(items[1:])

print(f"Written to {OUT_PARQUET}")

## 3. Read back and check if `storage:schemes` is preserved

In [ ]:
result = await rustac.read(OUT_PARQUET)
feat = result["features"][0]

print("Top-level keys after round-trip:", list(feat.keys()))
print("storage:schemes preserved:", "storage:schemes" in feat)
if "storage:schemes" in feat:
    print("storage:schemes:", feat["storage:schemes"])

## 4. Open a dataset directly with xpystac

If `storage:schemes` survived, `pystac.Item.from_dict()` will carry it into
`extra_fields` and xpystac can open the icechunk store without any reconstruction.

In [ ]:
ICECHUNK_MEDIA_TYPE = "application/vnd.zarr+icechunk"

client = rustac.DuckdbClient()
matches = list(client.search(OUT_PARQUET, limit=1))

item = pystac.Item.from_dict(matches[0])
print(f"Item: {item.id}")
print(f"storage:schemes: {item.extra_fields.get('storage:schemes')}")

ic_asset_key = next(
    k for k, a in item.assets.items() if a.media_type == ICECHUNK_MEDIA_TYPE
)
ds = xr.open_dataset(item.assets[ic_asset_key], engine="stac")
ds